# 加载包

In [1]:
import pandas as pd
import folium
from branca.colormap import LinearColormap

# 第二步：读取数据并聚合

In [ ]:
morning = pd.read_csv("/casual_weekday_7_9am_end_location.csv")
afternoon =  pd.read_csv("/casual_weekday_4_6pm_start-location.csv")

# 按网格分组计数
grid_morning = morning.groupby(['grid_lat', 'grid_lng']).size().reset_index(name='count')
grid_afternoon = afternoon.groupby(['grid_lat', 'grid_lng']).size().reset_index(name='count')


# 第三步：建立颜色映射

In [ ]:
#晚高峰的行程数量是早高峰的两倍多，所以要先算出晚高峰的 count 分布，作为统一标准

unified_vmax = grid_afternoon['count'].quantile(0.98)  # 用 98 分位数作为上限，避免极值影响颜色
unified_vmin = 1                                       # 固定下限为 1

print(f"统一颜色上限:{unified_vmax}")

#早高峰和晚高峰使用同一个 colormap
colormap = LinearColormap(
    colors=['#0d0887', '#7e03a8', '#cc4778', '#f89540', '#f0f921'],  # plasma 配色
    vmin=unified_vmin,
    vmax=unified_vmax  
)

统一颜色上限:4945.119999999989


# 第四步：定义函数，画地图

### 定义函数

In [ ]:
def plot_grid_map(data, m, colormap):
    # 每个网格画一个矩形（0.01度 x 0.01度）
    cell_size = 0.01                    

    for _, row in data.interrows():
         folium.Rectangle(
        bounds=[
            [row['grid_lat'],              row['grid_lng']],
            [row['grid_lat'] + cell_size,  row['grid_lng'] + cell_size]
        ],
        color=None,
        fill=True,
        fill_color=colormap(min(row['count'], unified_vmax)),              # 超出上限的按上限处理
        fill_opacity=min(0.55, 0.15 + row['count'] / unified_vmax * 0.4),  # 使用 unified_vmax
        tooltip=f"({row['grid_lat']}, {row['grid_lng']})<br>行程数：{row['count']:,}"
    ).add_to(m)

     # 填加图例
    colormap.caption = '行程数量'
    colormap.add_to(m)  

    return m


### 画地图

In [ ]:
# 早高峰

m_morning = folium.Map(
    location=[41.85, -87.65],
    zoom_start=11,
    tiles='OpenStreetMap'  
)

m_morning = plot_grid_map(grid_morning, m_morning, colormap)
m_morning.save("casual_commute_morning_end_heatmap.html")




# 晚高峰

m_afternoon = folium.Map(
    location=[41.85, -87.65],
    zoom_start=11,
    tiles='OpenStreetMap'  
)

m_afternoon = plot_grid_map(grid_afternoon, m_afternoon, colormap)
m_afternoon.save("casual_commute_afternoon_start_heatmap.html")